# MATH70113 – Simulation Methods for Finance
## Group Coursework — Jupyter Notebook

**Group members:**
| Name | CID |
|---|---|
| Paul Archer | 06054057 |
| Hamza El Arji | 06020926 |
| Chedi Mnif | 06064588 |
| Diane Murzi | 06048769 |

This notebook generates all numerical results and figures in the report.  
It is designed to run on **Google Colab with a T4 GPU (15 GB VRAM)** using [CuPy](https://cupy.dev/) for GPU acceleration. If no GPU is available, it falls back to NumPy automatically (slower but correct).

**Key implementation choices:**
- `xp` namespace — resolves to CuPy (GPU) or NumPy (CPU) at runtime
- `float32` arithmetic — halves memory use and roughly doubles GPU throughput vs `float64`
- Shared-Z sweeps — a single draw of Gaussian increments is reused across all parameter values in each sweep (×10 speedup)
- One-pass MSE (Part 2) — a single reference run at $N_{\text{ref}}$ paths; all columns of the MSE table follow from the exact identity $\text{MSE}(N) = \text{MSE}(N_{\text{ref}}) \times N_{\text{ref}} / N$
- `joblib.Parallel` — Longstaff–Schwartz trials are parallelised across CPU cores

**Estimated runtimes on a T4 GPU:**
| Section | Time |
|---|---|
| Part 1 (all) | ~5–10 s |
| Part 2 — MSE tables | ~15–20 s |
| Part 2 — LSM grid | ~45 s |
| **Total** | **~1 min** |

&nbsp;

> **Cells 14b and 14c are commented out.** They implement two additional convergence experiments (brute-force $N = 10^9$ run and Richardson extrapolation) that we ran separately to verify that the Brownian bridge bias lies below the Monte Carlo noise floor at all practical sample sizes. Cell 14b in particular takes $\approx$ 20 minutes even on a GPU and is included for reference only.

In [ ]:
import numpy as np
import math, time, warnings
from concurrent.futures import ProcessPoolExecutor
from scipy.stats import norm
from scipy.special import eval_laguerre, comb
from scipy.differentiate import derivative as _scipy_deriv
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LogNorm
import gc
import time
from joblib import Parallel, delayed
warnings.filterwarnings('ignore')

# GPU detection
try:
    import cupy as cp
    xp = cp
    # Warm up GPU
    _ = cp.array([1.0])
    props = cp.cuda.runtime.getDeviceProperties(0)
    mem   = cp.cuda.runtime.memGetInfo()
    print(f"GPU detected: {props['name'].decode()}")
    print(f"  VRAM: {props['totalGlobalMem']/1e9:.1f} GB  |  Free: {mem[0]/1e9:.1f} GB")
    DEVICE = 'GPU'
    # Chunk size for batched GPU operations — reduce if OOM
    GPU_CHUNK = min(200, int(mem[0] / (128_000 * 4 * 8)))  # auto-tune
    print(f"  Auto-tuned batch chunk: {GPU_CHUNK}")
except Exception as e:
    import numpy as cp   # cp becomes an alias for numpy
    xp = np
    DEVICE = 'CPU'
    GPU_CHUNK = 50
    print(f'No GPU / CuPy — running on CPU. (reason: {e})')
    print('Install CuPy: pip install cupy-cuda11x  (or cupy-cuda12x)')

def to_numpy(arr):
    """Transfer CuPy array to host NumPy array (no-op if already NumPy)."""
    return cp.asnumpy(arr) if DEVICE == 'GPU' else np.asarray(arr)

def f32(x):
    """Cast scalar to xp float32."""
    return xp.float32(x)

plt.rcParams.update({'figure.dpi': 130, 'font.size': 11,
                     'axes.grid': True, 'grid.alpha': 0.3})
SAVEFIG_KW = dict(dpi=150, bbox_inches='tight')
SEED = 42
print(f'\nRunning on {DEVICE}. All seeds fixed to {SEED}.')

---
# Part 1 — Pathwise Greeks for a Down-and-Out Barrier Call

We estimate $\Delta$ (Delta) and $\mathcal{V}$ (Vega) of a down-and-out barrier call option under GBM using the **pathwise sensitivity method** with a **Brownian bridge barrier correction**. All mathematical derivations are in the report (Section 2).

In [ ]:
r = 0.05;  sigma = 0.5;  T = 1.0
s0 = 100.0;  K_opt = 110.0;  B_base = 90.0

N_MAIN  = 500_000   # paths for the main run
N_SWEEP = 200_000   # paths for parameter sweeps (S0, sigma, B)
M_STEPS = 252       # daily discretisation

def num_deriv(f, x0, dx=1e-5):
    """
    Fifth-order central finite difference.
    Coefficients: [-1/12, 2/3, 0, -2/3, 1/12] / dx
    Error = O(dx^4) ~ 1e-20 for dx=1e-5 — negligible vs MC noise.
    Works with any scalar-valued function, no vectorisation required.
    """
    return (-f(x0 + 2*dx) + 8*f(x0 + dx) - 8*f(x0 - dx) + f(x0 - 2*dx)) / (12*dx)

def do_call_price(s, K, B, r, sigma, T):
    """Closed-form down-and-out barrier call price (Black-Scholes)."""
    lam = (r + 0.5*sigma**2) / sigma**2
    sqT = sigma * np.sqrt(T)
    x1  = np.log(s/K) / sqT + lam*sqT
    y1  = np.log(B**2/(s*K)) / sqT + lam*sqT
    return (s*norm.cdf(x1) - K*np.exp(-r*T)*norm.cdf(x1-sqT)
            - s*(B/s)**(2*lam)*norm.cdf(y1)
            + K*np.exp(-r*T)*(B/s)**(2*lam-2)*norm.cdf(y1-sqT))

eps_s, eps_v = 0.5, 0.002
V_exact     = do_call_price(s0, K_opt, B_base, r, sigma, T)
Delta_exact = num_deriv(lambda s: do_call_price(s, K_opt, B_base, r, sigma, T), s0)
Vega_exact  = num_deriv(lambda v: do_call_price(s0, K_opt, B_base, r, v, T),    sigma)

print(f'Exact price: {V_exact:.6f}  |  Delta: {Delta_exact:.6f}  |  Vega: {Vega_exact:.6f}')

In [ ]:
# Core MC function — GPU-ready via xp namespace + float32

def mc_barrier_pathwise(N, n, s0, r, sigma, T, B, K_s, seed=SEED,
                        _Z=None, _cumZ=None):
    """
    Pathwise Delta & Vega for a down-and-out barrier call.

    OPTIMISATION: if pre-generated (_Z, _cumZ) are supplied,
    they are reused — enables shared-Z sweeps (x10 speedup).

    Returns dict with price/delta/vega + standard errors.
    """
    h      = np.float32(T / n)
    sqrt_h = np.float32(np.sqrt(h))
    disc   = np.float32(np.exp(-r * T))
    lnB    = np.float32(np.log(B))
    s0f    = np.float32(s0)
    rf     = np.float32(r);  sf = np.float32(sigma)

    # Random numbers (reuse if supplied)
    if _Z is None:
        rng = xp.random.default_rng(seed) if hasattr(xp, 'random') else None
        if rng:
            _Z = rng.standard_normal((N, n)).astype(xp.float32)
        else:
            xp.random.seed(seed)
            _Z = xp.random.standard_normal((N, n)).astype(xp.float32)
    if _cumZ is None:
        _cumZ = xp.cumsum(_Z, axis=1)  # (N, n) — shared across sigma values

    # Simulate GBM paths
    log_inc = (rf - 0.5*sf*sf)*h + sf*sqrt_h*_Z   # (N, n)
    cumlog  = xp.cumsum(log_inc, axis=1)            # (N, n)
    logS    = xp.log(s0f) + cumlog                  # (N, n)
    S       = xp.exp(logS)                          # (N, n)

    logS_prev = xp.empty_like(logS)
    logS_prev[:, 0]  = xp.log(s0f)
    logS_prev[:, 1:] = logS[:, :-1]

    # Brownian-bridge survival weights
    l_prev     = logS_prev - lnB
    l_curr     = logS      - lnB
    both_above = (l_prev > 0) & (l_curr > 0)

    exp_arg = xp.float32(-2.0) * l_prev * l_curr / (sf*sf*h)
    p_cross = xp.where(both_above, xp.exp(exp_arg), xp.float32(1.0))
    q_surv  = xp.where(both_above, xp.float32(1.0) - p_cross, xp.float32(0.0))

    with np.errstate(divide='ignore'):
        log_q = xp.where(q_surv > 0,
                         xp.log(xp.maximum(q_surv, xp.float32(1e-38))),
                         xp.float32(-1e30))
    log_W = log_q.sum(axis=1)
    W     = xp.where(log_W > xp.float32(-1e29), xp.exp(log_W), xp.float32(0.0))

    # Payoff
    ST     = S[:, -1]
    payoff = xp.maximum(ST - xp.float32(K_s), xp.float32(0.0))
    itm    = (ST > xp.float32(K_s)).astype(xp.float32)
    f      = disc * payoff * W

   # Pathwise Delta (full product rule, both terms)
    dST_ds0     = ST / s0f
    dp_ds0      = xp.where(both_above,
                           p_cross * (xp.float32(-2.0)/(sf*sf*h)) * (l_curr + l_prev) / s0f,
                           xp.float32(0.0))
    with np.errstate(divide='ignore', invalid='ignore'):
        ratio_s = xp.where(q_surv > 0, -dp_ds0 / q_surv, xp.float32(0.0))
    dW_ds0       = W * ratio_s.sum(axis=1)
    delta_term1  = disc * itm * dST_ds0 * W          # term 1 only (biased if used alone)
    delta_term2  = disc * payoff * dW_ds0             # product-rule correction
    delta_paths  = delta_term1 + delta_term2

    # Pathwise Vega (full product rule, both terms)
    k_idx       = xp.arange(1, n+1, dtype=xp.float32)
    dlogS_dsig  = -sf * (k_idx * h) + sqrt_h * _cumZ
    dlogS_prev  = xp.empty_like(dlogS_dsig)
    dlogS_prev[:, 0]  = xp.float32(0.0)
    dlogS_prev[:, 1:] = dlogS_dsig[:, :-1]

    dST_dsig    = ST * dlogS_dsig[:, -1]
    dp_dsig     = xp.where(both_above,
        p_cross * (xp.float32(4.0)*l_prev*l_curr/(sf*sf*sf*h)
                   - (xp.float32(2.0)/(sf*sf*h)) *
                     (dlogS_prev*l_curr + l_prev*dlogS_dsig)),
        xp.float32(0.0))
    with np.errstate(divide='ignore', invalid='ignore'):
        ratio_v = xp.where(q_surv > 0, -dp_dsig / q_surv, xp.float32(0.0))
    dW_dsig     = W * ratio_v.sum(axis=1)
    vega_term1  = disc * itm * dST_dsig * W
    vega_paths  = vega_term1 + disc * payoff * dW_dsig

    def _stats(arr):
        a = to_numpy(arr).astype(np.float64)
        return float(a.mean()), float(a.std(ddof=1) / np.sqrt(N))

    p, pse = _stats(f)
    d, dse = _stats(delta_paths)
    v, vse = _stats(vega_paths)

    # term1_d / term1_v stored as means only (no memory overhead)
    return dict(price=p, price_se=pse,
                delta=d, delta_se=dse,
                vega=v,  vega_se=vse,
                term1_d=float(to_numpy(delta_term1).astype(np.float64).mean()),
                term1_v=float(to_numpy(vega_term1 ).astype(np.float64).mean()),
                f=f, delta_raw=delta_paths, vega_raw=vega_paths)


def mc_barrier_naive(N, n, s0, r, sigma, T, B, K_s, seed=SEED, _Z=None):
    h   = np.float32(T/n)

    if _Z is None:
        xp.random.seed(seed)
        Z = xp.random.standard_normal((N, n)).astype(xp.float32)
    else:
        Z = _Z

    logS = xp.log(xp.float32(s0)) + xp.cumsum(
        (np.float32(r)-np.float32(0.5*sigma**2))*h + np.float32(sigma)*np.float32(np.sqrt(h))*Z,
        axis=1)
    S    = xp.exp(logS)
    surv = (S > xp.float32(B)).all(axis=1).astype(xp.float32)
    f    = np.float32(np.exp(-r*T)) * xp.maximum(S[:,-1] - xp.float32(K_s), xp.float32(0.0)) * surv
    fa   = to_numpy(f).astype(np.float64)
    return float(fa.mean()), float(fa.std(ddof=1)/np.sqrt(N))


print('Part 1 MC functions defined (GPU-ready, float32, shared-Z capable).')

## 1.1  Reference Values and Main Simulation

The Black–Scholes closed-form price of a down-and-out barrier call is used as a reference. Delta and Vega references are obtained by fourth-order centred finite differences on this formula (step $h = 10^{-5}$, error $< 10^{-10}$).

The main run uses $N = 5 \times 10^5$ paths, $M = 252$ daily steps, seed 42.

In [ ]:
t0 = time.perf_counter()
res = mc_barrier_pathwise(N_MAIN, M_STEPS, s0, r, sigma, T, B_base, K_opt, seed=SEED)
t_main = time.perf_counter() - t0

print(f'Main run ({N_MAIN:,} paths, {M_STEPS} steps): {t_main:.2f}s  [{DEVICE}]')
print('='*66)
print(f"{'Quantity':<12} {'Exact/FD':>12} {'MC':>12} {'95% CI':>24}")
print('-'*66)
ci = lambda m, se: f'[{m-1.96*se:.5f}, {m+1.96*se:.5f}]'
for label, exact, mc_val, mc_se in [
    ('Price',  V_exact,     res['price'], res['price_se']),
    ('Delta',  Delta_exact, res['delta'], res['delta_se']),
    ('Vega',   Vega_exact,  res['vega'],  res['vega_se']),
]:
    print(f"{label:<12} {exact:>12.6f} {mc_val:>12.6f} {ci(mc_val, mc_se):>24}")
print('='*66)

## 1.2  Parameter Sweeps — Shared-Z Optimisation

All three sweeps ($S_0$, $\sigma$, $B$) reuse the **same matrix of Gaussian increments** $Z \in \mathbb{R}^{N \times M}$. This works because the log-price increment $\Delta \log S_i = (r - \tfrac{\sigma^2}{2})h + \sigma\sqrt{h}\,Z_i$ depends linearly on $Z_i$, so changing $s_0$ or $\sigma$ does not require re-drawing $Z$.

In [ ]:
# Pre-generate shared random numbers
t0 = time.perf_counter()

if hasattr(xp, 'random') and hasattr(xp.random, 'default_rng'):
    rng_sw = xp.random.default_rng(SEED)
    Z_shared   = rng_sw.standard_normal((N_SWEEP, M_STEPS)).astype(xp.float32)
else:
    xp.random.seed(SEED)
    Z_shared   = xp.random.standard_normal((N_SWEEP, M_STEPS)).astype(xp.float32)

cumZ_shared = xp.cumsum(Z_shared, axis=1)  # shape (N_SWEEP, M_STEPS)
t_gen = time.perf_counter() - t0
print(f'Shared Z generated ({N_SWEEP:,} paths, {M_STEPS} steps): {t_gen:.2f}s')

# S0 sweep
s0_vals = np.linspace(92, 150, 30)
t0 = time.perf_counter()
res_s0 = [mc_barrier_pathwise(N_SWEEP, M_STEPS, s, r, sigma, T, B_base, K_opt,
                               _Z=Z_shared, _cumZ=cumZ_shared)
          for s in s0_vals]
t_s0 = time.perf_counter() - t0
print(f'S0 sweep (30 values, shared Z): {t_s0:.2f}s  [{DEVICE}]')

# Sigma sweep
sigma_vals = np.linspace(0.15, 0.90, 25)
t0 = time.perf_counter()
res_sig = [mc_barrier_pathwise(N_SWEEP, M_STEPS, s0, r, sv, T, B_base, K_opt,
                                _Z=Z_shared, _cumZ=cumZ_shared)
           for sv in sigma_vals]
t_sig = time.perf_counter() - t0
print(f'Sigma sweep (25 values, shared Z): {t_sig:.2f}s  [{DEVICE}]')

# B sweep
B_vals = np.linspace(60, 97, 20)
t0 = time.perf_counter()
res_B = [mc_barrier_pathwise(N_SWEEP, M_STEPS, s0, r, sigma, T, b, K_opt,
                              _Z=Z_shared, _cumZ=cumZ_shared)
         for b in B_vals]
t_B = time.perf_counter() - t0
print(f'Barrier sweep (20 values, shared Z): {t_B:.2f}s  [{DEVICE}]')

## 1.3  Greeks vs Parameters

In [ ]:
# 1. Extract MC results from res_s0
delta_s0 = np.array([x['delta'] for x in res_s0])
delta_s0_se = np.array([x['delta_se'] for x in res_s0])
delta_s0_term1 = np.array([x['term1_d']  for x in res_s0])  # biased (term 1 only)
vega_s0 = np.array([x['vega'] for x in res_s0])
vega_s0_se = np.array([x['vega_se'] for x in res_s0])

# 2. Calculate exact analytical curves (via finite differences on BS formula)
delta_exact_curve = [
    num_deriv(lambda s_: do_call_price(s_, K_opt, B_base, r, sigma, T), s)
    for s in s0_vals
]

vega_exact_curve = [
    num_deriv(lambda v: do_call_price(s, K_opt, B_base, r, v, T), sigma)
    for s in s0_vals
]

delta_exact_B = [
    num_deriv(lambda s, b_=b: do_call_price(s, K_opt, b_, r, sigma, T), s0)
    for b in B_vals
]

# Figure 1: Delta vs S0 + Vega vs S0
fig1, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 1. Delta vs S0 (Top Left)
ax1.plot(s0_vals, delta_exact_curve, 'k-',   lw=2.5, label='Analytical (FD on exact formula)')
ax1.plot(s0_vals, delta_s0,          'b-o',  ms=4,   label='Pathwise (full, 2 terms)')
ax1.plot(s0_vals, delta_s0_term1,    'r--s', ms=3,   alpha=0.7,
         label='Pathwise (Term 1 only — biased)')
ax1.fill_between(s0_vals,
                 delta_s0 - 1.96*delta_s0_se,
                 delta_s0 + 1.96*delta_s0_se,
                 alpha=0.2, color='steelblue', label='95% CI (full)')
ax1.axvline(B_base, color='red',  ls=':', lw=1.5, label=f'B={B_base}')
ax1.axvline(K_opt,  color='gray', ls=':', lw=1.5, label=f'K={K_opt}')
ax1.set_xlabel('$S_0$');  ax1.set_ylabel('Delta')
ax1.set_title('Delta vs $S_0$');  ax1.legend(fontsize=8)

# ── Remark 2.1 verification: relative contribution of Term 2 at s0=100 ──────
# Find index closest to s0=100
d_full  = float(res['delta'])
d_term1 = float(res['term1_d'])
d_term2 = d_full - d_term1
frac    = 100.0 * d_term2 / d_full
print(f"Remark 1 check (N={N_MAIN:,}, s0={s0} exact):")
print(f"  Term 1 = {d_term1:.3f}")
print(f"  Term 2 = {d_term2:.3f}")
print(f"  Full Δ̂  = {d_full:.3f}  [same as Table 1]")
print(f"  Term 2 contributes {frac:.1f}% of Δ̂")

# 2. Vega vs S0 (Top Right) - The newly added plot
ax2.plot(s0_vals, vega_exact_curve, 'k-', lw=2.5, label='Analytical (exact)')
ax2.plot(s0_vals, vega_s0, 'g-o', ms=4, label='Pathwise (full)')
ax2.fill_between(s0_vals, vega_s0-1.96*vega_s0_se, vega_s0+1.96*vega_s0_se,
                alpha=0.2, color='green', label='95% CI')
ax2.axvline(B_base, color='red',  ls=':', lw=1.5, label=f'B={B_base}')
ax2.axvline(K_opt,  color='gray', ls=':', lw=1.5, label=f'K={K_opt}')
ax2.set_xlabel('$S_0$'); ax2.set_ylabel('Vega'); ax2.set_title('Vega vs $S_0$')
ax2.legend(fontsize=8)

fig1.suptitle(f'Pathwise Greeks vs $S_0$ — BB Correction [{DEVICE}]', fontweight='bold')
plt.tight_layout()
plt.savefig('greeks_s0.png', **SAVEFIG_KW);  plt.show()

# Figure 2: Vega vs sigma + Price/Delta vs B
fig2, (ax3, ax4) = plt.subplots(1, 2, figsize=(14, 5))

# 3. Vega vs sigma (Bottom Left)
vega_sig = np.array([x['vega'] for x in res_sig])
vega_sig_se = np.array([x['vega_se'] for x in res_sig])
vega_exact_sig = [(do_call_price(s0, K_opt, B_base, r, sv+eps_v, T)
                 - do_call_price(s0, K_opt, B_base, r, sv-eps_v, T))/(2*eps_v)
                  for sv in sigma_vals]
ax3.plot(sigma_vals, vega_exact_sig, 'k-', lw=2.5, label='Analytical')
ax3.plot(sigma_vals, vega_sig, 'g-o', ms=4, label='Pathwise')
ax3.fill_between(sigma_vals, vega_sig-1.96*vega_sig_se, vega_sig+1.96*vega_sig_se,
                alpha=0.2, color='green', label='95% CI')
ax3.axvline(sigma, color='gray', ls=':', lw=1.5)
ax3.set_xlabel('$\\sigma$'); ax3.set_ylabel('Vega'); ax3.set_title('Vega vs $\\sigma$')
ax3.legend(fontsize=8)

# 4. Delta vs Barrier B (Bottom Right)
delta_B = np.array([x['delta'] for x in res_B])
price_B = np.array([x['price'] for x in res_B])
price_B_exact = [do_call_price(s0, K_opt, b, r, sigma, T) for b in B_vals]
ax4.plot(B_vals, price_B_exact, 'k-', lw=2.5, label='Price (exact)')
ax4.plot(B_vals, price_B, 'b-o', ms=4, label='Price (MC)')
ax4b = ax4.twinx()
ax4b.plot(B_vals, delta_exact_B, 'k-', lw=2.5, label='Delta (exact)')
ax4b.plot(B_vals, delta_B, 'r-s', ms=4, label='Delta (MC)', alpha=0.7)
ax4b.set_ylabel('Delta', color='red')
ax4.set_xlabel('Barrier $B$'); ax4.set_ylabel('Price', color='steelblue')
ax4.set_title('Price & Delta vs Barrier $B$')
lines4,  labs4  = ax4.get_legend_handles_labels()
lines4b, labs4b = ax4b.get_legend_handles_labels()
ax4b.legend(lines4 + lines4b, labs4 + labs4b, fontsize=8, loc='center left')

fig2.suptitle(f'Pathwise Greeks — Parameter Sensitivity [{DEVICE}]', fontweight='bold')
plt.tight_layout()
plt.savefig('greeks_params.png', **SAVEFIG_KW);  plt.show()

## 1.4  Weak Convergence in $h$ and Statistical Convergence in $N$

Two estimators are compared as $h = T/M \to 0$:

- **Naive:** survival indicator $\mathbf{1}_{\min_i S_{t_i} > B}$ — ignores crossings between grid points, bias $= \mathcal{O}(\sqrt{h})$
- **BB:** soft survival weight $\mathcal{S} = \prod_i (1 - p_i)$ where $p_i = \exp\!\left(-\frac{2\ln(S_{t_i}/B)\ln(S_{t_{i+1}}/B)}{\sigma^2 h}\right)$ — bias $= \mathcal{O}(h)$

The BB bias falls below the MC noise floor at all practical $N$, so the $\mathcal{O}(h)$ slope cannot be resolved numerically without $N \gtrsim 10^{10}$ paths.

In [ ]:
#1.4 Weak convergence (bias vs h) and statistical convergence (SE vs N)
#
# We vary M (number of time steps, h = T/M) to measure how the discretisation
# bias shrinks as the grid gets finer. Two estimators are compared:
#
#   Naive : survival = 1{min_i S_{t_i} > B} — hard indicator on grid points only.
#           Misses barrier crossings between grid points → bias = O(sqrt(h)).
#
#   BB    : survival = prod_i (1 - p_i), where p_i is the analytically exact
#           probability that the continuous GBM crosses B in [t_i, t_{i+1}]
#           given the two endpoints (Brownian bridge formula).
#           Soft weight → bias = O(h), i.e. halving h halves the bias
#           (vs only a sqrt(2) reduction for Naive).
#
# Reference = analytical closed-form price/delta/vega (V_exact, Delta_exact,
# Vega_exact computed in Cell 3). Using the analytical formula as reference
# eliminates Monte Carlo noise from the reference entirely, giving a clean
# picture of discretisation bias only.
#
# Key finding: the BB correction is so effective that its residual O(h) bias
# falls below the Monte Carlo noise floor for all practical N. Resolving it
# would require N ~ 10^10 paths. The Naive bias, by contrast, is clearly
# detectable and follows O(sqrt(h)) as predicted by theory.
#
# Two loops are needed:
#   Loop 1 (bias vs h) : chunked over N to avoid GPU OOM.
#                        BB and Naive share the same per-chunk seed so their
#                        biases are compared on identical random draws,
#                        reducing comparison variance.
#   Loop 2 (SE vs N)   : small individual calls, no chunking needed.

M_vals   = [5, 10, 20, 50, 100, 200, 500]
N_wk     = 500_000   # noise floor ~SE/sqrt(N_wk) ~ 6e-5 for price — still above BB bias
CHUNK_WK = 100_000   # 100k × 500 steps × 17 arrays × 4B ≈ 3.4 GB VRAM — safe on 16 GB GPU
SEED_WK  = 99

if DEVICE == 'GPU':
    cp.get_default_memory_pool().free_all_blocks()

# Reference: analytical closed-form (zero Monte Carlo noise)
P_ref = V_exact       # exact Black-Scholes barrier call price
D_ref = Delta_exact   # num_deriv of closed-form, error < 1e-10
V_ref = Vega_exact    # num_deriv of closed-form, error < 1e-10
print(f"Analytical reference: price={P_ref:.6f} | delta={D_ref:.6f} | vega={V_ref:.6f}")

# Helper: chunked average of BB and Naive
def chunked_run(N_total, chunk, M, seed_base, run_naive=False):
    """
    Runs mc_barrier_pathwise (and optionally mc_barrier_naive) in blocks of
    `chunk` paths to avoid GPU OOM errors.

    The mean of equal-sized block means equals the global mean exactly,
    so this is statistically equivalent to a single run of N_total paths.

    BB and Naive share the same per-chunk seed so their bias estimates are
    compared on identical underlying random numbers, reducing comparison variance.

    Returns (price_bb, delta_bb, vega_bb, price_naive or None).
    """
    n_chunks = N_total // chunk
    p_acc = d_acc = v_acc = pn_acc = 0.0
    for i in range(n_chunks):
        seed_i  = seed_base + i * 31          # distinct independent seeds per chunk
        res_c   = mc_barrier_pathwise(chunk, M, s0, r, sigma, T,
                                      B_base, K_opt, seed=seed_i)
        p_acc  += res_c['price'] / n_chunks
        d_acc  += res_c['delta'] / n_chunks
        v_acc  += res_c['vega']  / n_chunks
        if run_naive:
            pn, _  = mc_barrier_naive(chunk, M, s0, r, sigma, T,
                                       B_base, K_opt, seed=seed_i)
            pn_acc += pn / n_chunks
        if DEVICE == 'GPU':
            cp.get_default_memory_pool().free_all_blocks()
    return p_acc, d_acc, v_acc, (pn_acc if run_naive else None)

# Loop 1: bias vs h
bias_bb    = {'price': [], 'delta': [], 'vega': []}
bias_naive = {'price': []}

print(f"\nWeak convergence sweep (N={N_wk:,}, chunk={CHUNK_WK:,}, "
      f"{N_wk // CHUNK_WK} chunks per M):")
for M in M_vals:
    p, d, v, pn = chunked_run(N_wk, CHUNK_WK, M, SEED_WK, run_naive=True)
    bias_bb['price'].append(abs(p  - P_ref))
    bias_bb['delta'].append(abs(d  - D_ref))
    bias_bb['vega'].append( abs(v  - V_ref))
    bias_naive['price'].append(abs(pn - P_ref))
    print(f"  M={M:4d} (h={T/M:.4f}) | "
          f"BB price={bias_bb['price'][-1]:.2e} | "
          f"Naive={bias_naive['price'][-1]:.2e} | "
          f"BB delta={bias_bb['delta'][-1]:.2e}")

# Loop 2: SE vs N at fixed M=252
# Each call is small (N_se <= 200k, M=252) — no chunking needed.
N_se_list = [1000, 2000, 5000, 10000, 25000, 50000, 100000, 200000]
se_d, se_v = [], []

for N_se in N_se_list:
    r_n = mc_barrier_pathwise(N_se, M_STEPS, s0, r, sigma, T,
                               B_base, K_opt, seed=SEED)
    se_d.append(r_n['delta_se'])
    se_v.append(r_n['vega_se'])
    if DEVICE == 'GPU':
        cp.get_default_memory_pool().free_all_blocks()

# Plotting
h_vals = [T / M for M in M_vals]
N_arr  = np.array(N_se_list, dtype=float)

# Monte Carlo noise floor for price at N=N_wk (from the main run)
# BB bias oscillates around this level — it cannot be resolved at any practical N
noise_floor_price = res['price_se']   # SE of main run (N=500k, M=252)
noise_floor_delta = res['delta_se']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left panel: weak convergence in h
ax = axes[0]
ax.loglog(h_vals, bias_bb['price'],    'b-o',  ms=5, label='Price (BB)')
ax.loglog(h_vals, bias_naive['price'], 'r--s', ms=5, label='Price (Naive)')
ax.loglog(h_vals, bias_bb['delta'],    'g-^',  ms=5, label='Delta (BB)')

# Noise floor: BB bias is undetectable below this level
ax.axhline(noise_floor_price, color='steelblue', ls=':', lw=1.5, alpha=0.8,
           label=f'MC noise floor (price, $N={N_wk:,}$): {noise_floor_price:.3f}')
ax.axhline(noise_floor_delta, color='green', ls=':', lw=1.5, alpha=0.8,
           label=f'MC noise floor (delta, $N={N_wk:,}$): {noise_floor_delta:.3f}')

# Reference slope for Naive only — BB is below noise floor so O(h) slope is not shown
c_naive = bias_naive['price'][2] / h_vals[2]**0.5
ax.loglog(h_vals, [c_naive * h**0.5 for h in h_vals], 'k--', lw=1.5,
          label='$O(\\sqrt{h})$ rate')

ax.set_xlabel('Step size $h = T/M$')
ax.set_ylabel('|Bias|')
ax.set_title('Weak Convergence: Naive $O(\\sqrt{h})$, BB below noise floor')
ax.legend(fontsize=7.5)

# Right panel: statistical convergence in N
ax = axes[1]
ax.loglog(N_arr, se_d, 'r-o', ms=5, label='SE(Delta)')
ax.loglog(N_arr, se_v, 'g-s', ms=5, label='SE(Vega)')

# Reference line calibrated on the largest N point of SE(Delta)
ref_se = se_d[-1] * np.sqrt(N_arr[-1]) / np.sqrt(N_arr)
ax.loglog(N_arr, ref_se, 'k--', lw=1.5, label='$O(N^{-1/2})$ rate')

ax.set_xlabel('$N$ (paths)')
ax.set_ylabel('Standard Error')
ax.set_title('SE vs $N$ — confirms $O(N^{-1/2})$ rate')
ax.legend()

fig.suptitle('Part 1 Convergence Analysis', fontweight='bold')
plt.tight_layout()
plt.savefig('weak_convergence.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: weak_convergence.png')

---

**Cells 14b and 14c** (below) implement the brute-force $N = 10^9$ convergence run and Richardson extrapolation used to further verify the Brownian bridge $\mathcal{O}(h)$ rate. Both cells are commented out: cell 14b takes $\approx$ 20 minutes to complete on T4 GPU. Results are discussed in Section 2.4 of the report.

In [ ]:
# Extremely slow to run cell (N=10^9)

# 1.4b High-N Convergence
# Optimisations: Shared random variables, adaptive memory chunking (target ~8GB),
# persistent RNG instances, and deferred garbage collection.

#M_coarse = [2, 3, 5, 8, 12, 20, 50, 252]
#N_LARGE  = 1_000_000_000   # 10^9
#SEED_LG  = 777
#
#def get_chunk_size_opt(M):
#    """
#    Determines optimal chunk size to maximize GPU VRAM utilization (~8GB peak)
#    and minimize host-to-device synchronization overhead.
#    """
#    target_bytes = 8_000_000_000
#    n_arrays     = 17
#    chunk        = target_bytes // (M * n_arrays * 4)
#    return int(np.clip(chunk, 100_000, 40_000_000))
#
## Empirical noise floor estimation
#sigma_price = float(res['price_se']) * np.sqrt(N_MAIN)
#noise_lg    = sigma_price / np.sqrt(N_LARGE)
#
#print("High-N Convergence Analysis")
#print(f"  N           = {N_LARGE:,}")
#print(f"  Noise floor ~ {noise_lg:.2e}")
#print(f"  V_exact     = {V_exact:.6f}\n")
#
#print("Batch Configuration:")
#for M in M_coarse:
#    c = get_chunk_size_opt(M)
#    print(f"  M={M:3d}  chunk={c:>10,}  n_chunks={N_LARGE//c:>6,}"
#          f"  peak_mem~{M*c*17*4/1e9:.2f}GB")
#print()
#
#print(f"{'M':>5} {'h':>8} {'|bias_BB|':>12} {'|bias_Naive|':>13} "
#      f"{'ratio':>7} {'detectable':>12} {'time':>7}")
#print('-'*75)
#
#bias_bb_lg    = []
#bias_naive_lg = []
#t0_total = time.perf_counter()
#
#for M in M_coarse:
#    h          = T / M
#    chunk_sz   = get_chunk_size_opt(M)
#    n_chunks_i = N_LARGE // chunk_sz
#    t0m        = time.perf_counter()
#
#    acc_bb    = 0.0
#    acc_naive = 0.0
#
#    # Persistent RNG initialization per discretization level
#    if hasattr(xp, 'random') and hasattr(xp.random, 'default_rng'):
#        rng_lg = xp.random.default_rng(SEED_LG + M * 1000)
#        def gen_Z(N, n):
#            return rng_lg.standard_normal((N, n), dtype=xp.float32)
#    else:
#        xp.random.seed(SEED_LG + M * 1000)
#        def gen_Z(N, n):
#            return xp.random.standard_normal((N, n)).astype(xp.float32)
#
#    for i in range(n_chunks_i):
#        # Generate shared standard normal increments
#        Z_chunk = gen_Z(chunk_sz, M)
#
#        # Brownian Bridge estimator
#        res_bb = mc_barrier_pathwise(
#            chunk_sz, M, s0, r, sigma, T, B_base, K_opt,
#            seed=SEED_LG, _Z=Z_chunk
#        )
#        acc_bb += res_bb['price'] / n_chunks_i
#
#        # Naive estimator
#        price_nv, _ = mc_barrier_naive(
#            chunk_sz, M, s0, r, sigma, T, B_base, K_opt,
#            seed=SEED_LG, _Z=Z_chunk
#        )
#        acc_naive += price_nv / n_chunks_i
#
#        # Explicit reference deletion to assist standard memory management
#        del Z_chunk
#        del res_bb
#
#    b_bb  = abs(acc_bb    - V_exact)
#    b_nv  = abs(acc_naive - V_exact)
#    bias_bb_lg.append(b_bb)
#    bias_naive_lg.append(b_nv)
#
#    ratio      = b_bb / noise_lg
#    detectable = "YES ✓" if ratio > 3 else "no (< 3σ)"
#    elapsed    = time.perf_counter() - t0m
#
#    print(f"{M:5d} {h:8.4f} {b_bb:12.2e} {b_nv:13.2e} "
#          f"{ratio:7.1f}x {detectable:>12}  {elapsed:6.1f}s")
#
#    # Deferred garbage collection and GPU cache clearing
#    gc.collect()
#    if DEVICE == 'GPU':
#        cp.get_default_memory_pool().free_all_blocks()
#
#print('-'*75)
#print(f"Total: {time.perf_counter()-t0_total:.1f}s  [{DEVICE}]")
#
## Plotting
#h_vals = np.array([T / M for M in M_coarse])
#bb_arr = np.array(bias_bb_lg)
#nv_arr = np.array(bias_naive_lg)
#
#fig, ax = plt.subplots(figsize=(9, 5))
#
#ax.loglog(h_vals, nv_arr, 'r-s', ms=6, lw=2, label='Naive price bias')
#ax.loglog(h_vals, bb_arr, 'b-o', ms=6, lw=2, label='BB price bias')
#ax.axhline(noise_lg,     color='gray', ls=':',  lw=1.8,
#           label=f'MC noise floor ($N=10^9$): {noise_lg:.1e}')
#ax.axhline(3*noise_lg,   color='gray', ls='--', lw=1.2, alpha=0.5,
#           label=f'$3\\sigma$ threshold: {3*noise_lg:.1e}')
#
## Asymptotic reference rates
#ax.loglog(h_vals, nv_arr[0] * np.sqrt(h_vals / h_vals[0]),
#          'r--', lw=1.3, alpha=0.6, label=r'$O(\sqrt{h})$ reference')
#
#detectable_mask = bb_arr > 3 * noise_lg
#if detectable_mask.any():
#    idx_anc = int(np.argmax(bb_arr[detectable_mask]))
#    h_anc   = h_vals[detectable_mask][idx_anc]
#    b_anc   = bb_arr[detectable_mask][idx_anc]
#    ax.loglog(h_vals, b_anc * (h_vals / h_anc),
#              'b--', lw=1.3, alpha=0.6, label=r'$O(h)$ reference')
#    slope_msg = r"$O(h)$ slope detectable for coarse grids"
#else:
#    slope_msg = r"BB bias below noise floor for all $M$ — $O(h)$ slope unresolvable at $N=10^9$"
#
#ax.set_xlabel('Step size $h = T/M$', fontsize=11)
#ax.set_ylabel('|Price Bias|', fontsize=11)
#ax.set_title(f'BB vs Naive Convergence — $N=10^9$ paths [{DEVICE}]\n{slope_msg}',
#             fontsize=10)
#ax.legend(fontsize=8, loc='upper left')
#plt.tight_layout()
#plt.savefig('weak_convergence_highN.png', **SAVEFIG_KW)
#plt.show()
#print('Saved: weak_convergence_highN.png')
#
## Summary Output
#print(f"\nNoise floor={noise_lg:.2e},  3σ={3*noise_lg:.2e}")
#print(f"{'M':>5} {'h':>8} {'|bias_BB|':>12} {'bias/noise':>12} {'detectable':>12}")
#print('-'*55)
#for M, h, b in zip(M_coarse, h_vals, bb_arr):
#    r = b / noise_lg
#    print(f"{M:5d} {h:8.4f} {b:12.2e} {r:12.1f}x  {'YES' if r>3 else 'no'}")

In [ ]:
# 1.4c  Richardson extrapolation to isolate O(h) BB bias
#
# Key idea: simulate paths at resolution M and 2M on the SAME Brownian
# increments. BB(M) - BB(2M) cancels MC noise (correlated paths) and
# isolates the O(h) discretisation bias.
#
# BB(M)  - V_exact = C*h   + MC_noise(N)
# BB(2M) - V_exact = C*h/2 + MC_noise(N)   [same noise, correlated]
# => BB(M) - BB(2M) = C*h/2  [noise largely cancels]

#M_base_list = [2, 4, 8, 16, 32, 64]   # BB(M) vs BB(2M)
#N_RICH      = 10_000_000              # 10^7 suffices since noise cancels
#SEED_RC     = 555
#
#def refine_Z(Z_coarse):
#    """
#    Given Z of shape (N, M) for coarse grid,
#    build Z_fine of shape (N, 2M) for the fine grid
#    by splitting each increment into two half-increments.
#    Z_coarse[:,i] = Z_fine[:,2i] + Z_fine[:,2i+1]  (in distribution)
#    We use antithetic: Z_fine[:,2i] = Z_coarse[:,i]/2 + eps/2
#                       Z_fine[:,2i+1] = Z_coarse[:,i]/2 - eps/2
#    where eps ~ N(0,1) independent.
#    This ensures Z_fine[:,2i]+Z_fine[:,2i+1] = Z_coarse[:,i] exactly.
#    """
#    N, M = Z_coarse.shape
#    eps  = xp.random.standard_normal((N, M)).astype(xp.float32) / xp.float32(np.sqrt(2))
#    Z_fine = xp.empty((N, 2*M), dtype=xp.float32)
#    Z_fine[:, 0::2] = Z_coarse / xp.float32(np.sqrt(2)) + eps
#    Z_fine[:, 1::2] = Z_coarse / xp.float32(np.sqrt(2)) - eps
#    return Z_fine
#
#sigma_price = float(res['price_se']) * np.sqrt(N_MAIN)
#noise_rich  = sigma_price / np.sqrt(N_RICH)   # noise on each estimator
## Noise on the DIFFERENCE BB(M)-BB(2M): much smaller due to correlation
## In practice ~5-10x smaller than individual noise
#
#print(f"Richardson extrapolation: N={N_RICH:,}")
#print(f"Individual noise floor : {noise_rich:.2e}")
#print(f"Expected noise on diff : ~{noise_rich/5:.2e}  (correlated paths)")
#print()
#print(f"{'M':>5} {'h':>7} {'BB(M)-V':>12} {'BB(2M)-V':>12} "
#      f"{'diff=C*h/2':>12} {'C_est':>10} {'time':>7}")
#print('-'*75)
#
#diffs    = []
#h_halves = []
#C_estimates = []
#t0_rc = time.perf_counter()
#
#for M in M_base_list:
#    h    = T / M
#    t0m  = time.perf_counter()
#
#    acc_coarse = 0.0   # BB(M)
#    acc_fine   = 0.0   # BB(2M)
#    n_chunks_rc = max(1, N_RICH // 500_000)
#    chunk_rc    = N_RICH // n_chunks_rc
#
#    if hasattr(xp, 'random') and hasattr(xp.random, 'default_rng'):
#        rng_rc = xp.random.default_rng(SEED_RC + M * 100)
#        def gen_rc(N, n): return rng_rc.standard_normal((N,n)).astype(xp.float32)
#    else:
#        xp.random.seed(SEED_RC + M * 100)
#        def gen_rc(N, n): return xp.random.standard_normal((N,n)).astype(xp.float32)
#
#    for i in range(n_chunks_rc):
#        # Coarse grid Z: shape (chunk, M)
#        Z_c = gen_rc(chunk_rc, M)
#
#        # Fine grid Z: shape (chunk, 2M) — shares same BM increments
#        Z_f = refine_Z(Z_c)
#
#        # BB at coarse resolution M
#        res_c = mc_barrier_pathwise(
#            chunk_rc, M, s0, r, sigma, T, B_base, K_opt,
#            seed=SEED_RC, _Z=Z_c
#        )
#        acc_coarse += res_c['price'] / n_chunks_rc
#
#        # BB at fine resolution 2M — same underlying paths
#        res_f = mc_barrier_pathwise(
#            chunk_rc, 2*M, s0, r, sigma, T, B_base, K_opt,
#            seed=SEED_RC, _Z=Z_f
#        )
#        acc_fine += res_f['price'] / n_chunks_rc
#
#        del Z_c, Z_f
#        if DEVICE == 'GPU':
#            cp.get_default_memory_pool().free_all_blocks()
#
#    diff  = acc_coarse - acc_fine          # estimates C * h/2
#    C_est = diff / (h / 2)                 # recover constant C
#    diffs.append(diff)
#    h_halves.append(h / 2)
#    C_estimates.append(C_est)
#
#    elapsed = time.perf_counter() - t0m
#    print(f"{M:5d} {h:7.4f} {acc_coarse-V_exact:+12.4e} {acc_fine-V_exact:+12.4e} "
#          f"{diff:+12.4e} {C_est:+10.4f}  {elapsed:6.1f}s")
#
#    gc.collect()
#    if DEVICE == 'GPU':
#        cp.get_default_memory_pool().free_all_blocks()
#
#print('-'*75)
#print(f"Total: {time.perf_counter()-t0_rc:.1f}s  [{DEVICE}]")
#print(f"\nMean C estimate: {np.mean(C_estimates):.4f}  "
#      f"Std: {np.std(C_estimates):.4f}")
#
## ── Plot ──────────────────────────────────────────────────────────────────────
#h_arr   = np.array([T/M for M in M_base_list])
#diff_arr = np.array(diffs)
#C_mean   = np.mean(C_estimates)
#
#fig, axes = plt.subplots(1, 2, figsize=(13, 5))
#
## Left: BB(M) - BB(2M) vs h/2
#ax = axes[0]
#ax.plot(h_arr/2, diff_arr, 'b-o', ms=6, lw=2, label=r'$\mathrm{BB}(M) - \mathrm{BB}(2M)$')
#h_fit = np.linspace(h_arr.min()/2, h_arr.max()/2, 100)
#ax.plot(h_fit, C_mean * h_fit, 'k--', lw=1.5,
#        label=f'$O(h)$ fit: $C \\approx {C_mean:.3f}$')
#ax.axhline(0, color='gray', ls=':', lw=1)
#ax.set_xlabel('$h/2 = T/(2M)$')
#ax.set_ylabel(r'$\mathrm{BB}(M) - \mathrm{BB}(2M)$')
#ax.set_title('Richardson: isolating $O(h)$ BB bias')
#ax.legend(fontsize=9)
#
## Right: C estimate vs M
#ax = axes[1]
#ax.semilogx([T/M for M in M_base_list], C_estimates, 'rs-', ms=6, lw=2)
#ax.axhline(C_mean, color='k', ls='--', lw=1.5,
#           label=f'Mean $C = {C_mean:.4f}$')
#ax.set_xlabel('$M$ (number of steps)')
#ax.set_ylabel('Estimated $C$ in BB bias $= C \\cdot h$')
#ax.set_title('Stability of $O(h)$ constant across grids')
#ax.legend(fontsize=9)
#
#fig.suptitle(f'Richardson Extrapolation — BB $O(h)$ bias isolation '
#             f'[$N={N_RICH:,}$, {DEVICE}]', fontweight='bold')
#plt.tight_layout()
#plt.savefig('weak_convergence_richardson.png', **SAVEFIG_KW)
#plt.show()
#print('Saved: weak_convergence_richardson.png')

## 1.5  Per-Path Distributions and Bump-and-Revalue Cross-Check

Histograms of the per-path Delta and Vega estimators illustrate the heavy-tailed distributions (excess kurtosis $\approx 270$ for Delta, $\approx 125$ for Vega) that cause Vega's wider confidence interval despite equal sample size. A bump-and-revalue estimate (centred finite difference on the MC price, same seed for up/down bumps) is computed as a cross-check on Delta.

In [ ]:
delta_raw = to_numpy(res['delta_raw']).astype(np.float64)
vega_raw  = to_numpy(res['vega_raw']).astype(np.float64)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, arr, label, color in zip(
    axes, [delta_raw, vega_raw], ['Delta', 'Vega'], ['steelblue', 'green']
):
    lo, hi = np.percentile(arr, [0.5, 99.5])
    ax.hist(arr, bins=120, range=(lo, hi), color=color, alpha=0.7, density=True, log=True)
    ax.axvline(arr.mean(), color='red', lw=2, label=f'Mean = {arr.mean():.4f}')
    ax.axvline(0, color='black', lw=1, ls='--')
    kurt = float(np.mean((arr - arr.mean())**4) / arr.std()**4)
    ax.set_title(f'{label} distribution (kurtosis = {kurt:.1f})')
    ax.set_xlabel(f'Per-path {label}');  ax.legend(fontsize=9)

fig.suptitle('Per-Path Greek Distributions — Heavy Vega Tails', fontweight='bold')
plt.tight_layout()
plt.savefig('greek_histograms.png', **SAVEFIG_KW);  plt.show()

# Bump-and-revalue — same seed for r_up and r_dn → shared Z → reduced variance ──────
h_brv = 1.0

r_up = mc_barrier_pathwise(N_MAIN, M_STEPS, s0+h_brv, r, sigma, T, B_base, K_opt, seed=SEED+1000)
r_dn = mc_barrier_pathwise(N_MAIN, M_STEPS, s0-h_brv, r, sigma, T, B_base, K_opt, seed=SEED+1000)

delta_brv    = (r_up['price'] - r_dn['price']) / (2*h_brv)
delta_brv_se = np.sqrt(r_up['price_se']**2 + r_dn['price_se']**2) / (2*h_brv)

if DEVICE == 'GPU':
    cp.get_default_memory_pool().free_all_blocks()

print(f"\nDelta comparison (N={N_MAIN:,}):")
print(f"  Pathwise (2-term):  {res['delta']:.5f}  SE={res['delta_se']:.5f}")
print(f"  Bump-and-revalue:   {delta_brv:.5f}  SE={delta_brv_se:.5f}")
print(f"  Analytical (exact): {Delta_exact:.5f}")
eff = (res['delta_se'] / delta_brv_se)**2 * 2
print(f"  Efficiency ratio PW/BRV (equal cost): {eff:.3f}  (<1 means PW is better)")

In [ ]:
r_test_up = mc_barrier_pathwise(N_MAIN, M_STEPS, s0+h_brv, r, sigma, T, B_base, K_opt, seed=SEED+1000)
r_test_dn = mc_barrier_pathwise(N_MAIN, M_STEPS, s0-h_brv, r, sigma, T, B_base, K_opt, seed=SEED+1000)
print(f"price_up = {r_test_up['price']:.5f}  (attendu ~9.4)")
print(f"price_dn = {r_test_dn['price']:.5f}  (attendu ~7.7)")

---
# Part 2 — Number of Paths vs Number of Basis Functions (Glasserman & Yu 2004)

We reproduce the main numerical experiments of Glasserman & Yu (2004), which establish a sharp phase transition for regression-based Monte Carlo: MSE decays as $1/N$ below a critical threshold $K^*$ and diverges above it. All theoretical background is in the report (Section 3).

In [ ]:
# LSM and Binomial benchmark (CPU — backward induction is inherently sequential)

def simulate_gbm_paths(S0, r, sigma, T, m, N, seed=42):
    """CPU GBM simulation for LSM (returns numpy)."""
    rng = np.random.default_rng(seed)
    dt  = T / m
    Z   = rng.standard_normal((N, m)).astype(np.float32)
    logS = (np.log(np.float32(S0))
            + np.cumsum((np.float32(r-0.5*sigma**2)*np.float32(dt)
                        + np.float32(sigma*np.sqrt(dt))*Z), axis=1))
    return np.column_stack([np.full(N, S0, dtype=np.float32), np.exp(logS)])

def weighted_laguerre_basis(S, K):
    w = np.exp(-S / 2)
    return np.column_stack([w * eval_laguerre(k, S) for k in range(K)])

def lsm_american_put(S0, K_s, r, sigma, T, m, N, K_basis, seed=42):
    """Longstaff-Schwartz MC for American put."""
    S       = simulate_gbm_paths(S0, r, sigma, T, m, N, seed=seed)
    payoff  = np.maximum(K_s - S, 0.0)
    tau     = np.full(N, m, dtype=np.int32)
    disc    = np.float32(np.exp(-r * T / m))
    for t in range(m-1, 0, -1):
        itm   = payoff[:, t] > 0
        n_itm = np.sum(itm)
        if n_itm < K_basis + 1: continue
        future = payoff[itm, tau[itm]] * disc**(tau[itm] - t)
        X      = weighted_laguerre_basis(S[itm, t], K_basis)
        try:
            c  = np.linalg.lstsq(X, future, rcond=None)[0]
            ex = payoff[itm, t] > X @ c
        except np.linalg.LinAlgError: continue
        tau[np.where(itm)[0][ex]] = t
    prices = payoff[np.arange(N), tau] * disc**tau
    return float(prices.mean()), float(prices.std(ddof=1) / np.sqrt(N))

def binomial_american_put(S0, K, r, sigma, T, n_steps=5000):
    dt   = T / n_steps;  u = np.exp(sigma*np.sqrt(dt));  d = 1.0/u
    p    = (np.exp(r*dt) - d) / (u - d);  disc = np.exp(-r*dt)
    j    = np.arange(n_steps+1)
    V    = np.maximum(K - S0*u**(n_steps-j)*d**j, 0.0)
    for i in range(n_steps-1, -1, -1):
        j   = np.arange(i+1);  S_i = S0*u**(i-j)*d**j
        V   = np.maximum(disc*(p*V[:i+1]+(1-p)*V[1:i+2]), np.maximum(K-S_i, 0.0))
    return V[0]

print('LSM functions defined.')

In [ ]:
# ── Reproduce Longstaff-Schwartz (2001) Table 1 ──────────────────────────────
K_s=40; r_ls=0.06; sig_ls=0.2; m_ls=50

print(f"{'S0':>4} {'T':>4} {'Binomial':>10} {'LSM':>10} {'SE':>8} {'Bias':>8}")
print('='*48)
for T_ls in [1.0, 2.0]:
    for S0_v in [36, 38, 40, 42, 44]:
        binom = binomial_american_put(S0_v, K_s, r_ls, sig_ls, T_ls)
        lsm_p, lsm_se = lsm_american_put(S0_v, K_s, r_ls, sig_ls, T_ls, m_ls, 100_000, 4, seed=42)
        print(f'{S0_v:4d} {T_ls:4.1f} {binom:10.4f} {lsm_p:10.4f} {lsm_se:8.4f} {lsm_p-binom:+8.4f}')
    print()

## 2.1  MSE Table — Brownian Setting ($t_1 = 1$, $t_2 = 2$)

Basis: probabilistic Hermite polynomials $\psi_{nk}(x) = \frac{1}{\sqrt{k!}} \mathrm{He}_k\!\left(\frac{x}{\sqrt{t_n}}\right)$, which satisfy $\Psi = I$.

Worst-case target: $Y^* = \rho^{K/2} \psi_{2K}(S_2)$, $\rho = t_2/t_1 = 2$. Critical constant: $c_\rho = 2\log(2 + \sqrt{\rho}) \approx 2.456$.

We use the **one-pass** method: estimate $\mathrm{Var}(\hat\gamma)$ once at $N_{\text{ref}}$, then recover $\mathrm{MSE}(N) = \mathrm{MSE}(N_{\text{ref}}) \times N_{\text{ref}} / N$ (exact, since the estimator is unbiased and paths are i.i.d.).

In [ ]:
def hermite_he(n, x):
    """Probabilist Hermite polynomial, vectorised over x (xp array)."""
    if n == 0: return xp.ones_like(x)
    if n == 1: return x.copy()
    h2, h1 = xp.ones_like(x), x.copy()
    for k in range(2, n+1):
        hc = x*h1 - f32(k-1)*h2;  h2, h1 = h1, hc
    return h1

def psi_hermite(k, x, t):
    return hermite_he(k, x / f32(np.sqrt(t))) / f32(math.sqrt(math.factorial(k)))

def lemma1_analytical(K, rho):
    return float(sum(rho**(-k) * comb(2*k,k,exact=True) * comb(K,k,exact=True)**2
                     for k in range(K+1)))

t1_gy, t2_gy = 1.0, 2.0
rho_gy  = t2_gy / t1_gy
c_rho   = 2 * np.log(2 + np.sqrt(rho_gy))
print(f'c_rho = 2 log(2+sqrt({rho_gy})) = {c_rho:.6f}')

# MC verification (on GPU)
K_check = range(1, 9)
moments_analytical, moments_mc = [], []

N_lem = 500_000
if hasattr(xp, 'random') and hasattr(xp.random, 'default_rng'):
    rng_lem = xp.random.default_rng(42)
    S1_lem = (rng_lem.standard_normal(N_lem) * float(np.sqrt(t1_gy))).astype(xp.float32)
    S2_lem = S1_lem + (rng_lem.standard_normal(N_lem) * float(np.sqrt(t2_gy-t1_gy))).astype(xp.float32)
else:
    xp.random.seed(42)
    S1_lem = xp.random.normal(0, float(np.sqrt(t1_gy)), N_lem).astype(xp.float32)
    S2_lem = S1_lem + xp.random.normal(0, float(np.sqrt(t2_gy-t1_gy)), N_lem).astype(xp.float32)

print(f'\n{"K":>3} {"Analytical":>15} {"MC":>15} {"Rel.err":>10} {"exp(c*K)":>15}')
print('-'*62)
for K in K_check:
    ma = lemma1_analytical(K, rho_gy)
    mm = float(xp.mean(psi_hermite(K, S2_lem, t2_gy)**2 * psi_hermite(K, S1_lem, t1_gy)**2))
    moments_analytical.append(ma)
    moments_mc.append(mm)
    print(f'{K:3d} {ma:15.4f} {mm:15.4f} {abs(mm-ma)/ma:10.4f} {np.exp(c_rho*K):15.2f}')

del S1_lem, S2_lem  # free memory

In [ ]:
def compute_mse_allN(K, t1, t2, N_ref, N_vals, B=5000, chunk_b=None):
    """
    GPU-accelerated MSE for ALL N values in one pass at N_ref.

    Trick: gamma_hat is a mean of N iid samples.
    Var(gamma_hat) = sigma^2_gamma / N_ref.
    MSE at any N = (N_ref / N) * Var(gamma_hat_at_N_ref).

    Processing: chunk_b batches at a time on GPU to control VRAM.
    """
    if chunk_b is None:
        # Auto-tune chunk size based on available memory
        chunk_b = GPU_CHUNK if DEVICE == 'GPU' else 100

    rho    = f32(t2 / t1)
    sq_t1  = f32(np.sqrt(t1));  sq_t2  = f32(np.sqrt(t2))
    sq_dt  = f32(np.sqrt(t2 - t1))
    sf     = xp.array([math.sqrt(math.factorial(k)) for k in range(K+1)], dtype=xp.float32)
    Yscale = f32(rho**(K/2))
    bt     = xp.zeros(K+1, dtype=xp.float32);  bt[K] = f32(1.0)

    # Accumulate sum and sum^2 of gamma_hat components over batches
    sum_g  = xp.zeros(K+1, dtype=xp.float64)
    sum_g2 = xp.zeros(K+1, dtype=xp.float64)

    batches_done = 0
    if hasattr(xp, 'random') and hasattr(xp.random, 'default_rng'):
        rng_m = xp.random.default_rng(42 + K)
        def gen(shape): return rng_m.standard_normal(shape).astype(xp.float32)
    else:
        xp.random.seed(42 + K)
        def gen(shape): return xp.random.standard_normal(shape).astype(xp.float32)

    while batches_done < B:
        b = min(chunk_b, B - batches_done)

        # (b, N_ref) float32
        S1c = gen((b, N_ref)) * sq_t1                           # S1 ~ N(0, t1)
        S2c = S1c + gen((b, N_ref)) * sq_dt                     # S2 - S1 ~ N(0, t2-t1)

        # Worst-case target Y* = rho^{K/2} * He_K(S2/sqrt(t2)) / sqrt(K!)
        Y   = Yscale * hermite_he(K, S2c / sq_t2) / sf[K]      # (b, N_ref)

        # gamma_hat[:,k] = mean_{N_ref}(Y * psi_k(S1)) for each batch
        for k in range(K+1):
            psi_k     = hermite_he(k, S1c / sq_t1) / sf[k]     # (b, N_ref)
            ghat_k    = (Y * psi_k).mean(axis=1).astype(xp.float64)  # (b,)
            sum_g[k]  += ghat_k.sum()
            sum_g2[k] += (ghat_k * ghat_k).sum()

        batches_done += b

    # Sample variance of gamma_hat[k] across B batches
    mean_g = sum_g / B
    var_g  = sum_g2 / B - mean_g**2     # E[gamma_hat^2] - (E[gamma_hat])^2

    # MSE(N) = (N_ref / N) * Var(gamma_hat) + squared_bias
    # Squared bias ≈ (gamma_hat_mean - beta_true)^2 (estimator is consistent)
    bt_np  = to_numpy(bt).astype(np.float64)
    vg_np  = to_numpy(var_g)
    mg_np  = to_numpy(mean_g)

    mse_dict = {}
    for N in N_vals:
        scale      = N_ref / N
        mse_dict[N] = float((scale * vg_np + (mg_np - bt_np)**2).sum())
    return mse_dict

print('Optimised MSE function defined.')

In [ ]:
N_vals_bm = [500, 1000, 2000, 4000, 8000, 16000, 32000, 64000, 128000]
K_vals_bm = list(range(1, 10))

# N_ref: compute at this N, scale to all others
# For K<=6: N_ref=128000 (in the regime where all N_vals can be computed)
# For K>=7: N_ref=500000 (same as original paper)
N_REF_SMALL = 128_000
N_REF_LARGE = 500_000
B_BATCHES   = 5000   # more than paper (was 5000 too, but now fast)

table1 = np.zeros((len(K_vals_bm), len(N_vals_bm)))

t0_table = time.perf_counter()
for ki, K in enumerate(K_vals_bm):
    t0 = time.perf_counter()
    N_ref = N_REF_SMALL if K <= 6 else N_REF_LARGE
    mse_d = compute_mse_allN(K, t1_gy, t2_gy, N_ref, N_vals_bm, B=B_BATCHES)
    for ni, N in enumerate(N_vals_bm):
        table1[ki, ni] = mse_d[N]
    print(f'  K={K:2d} (N_ref={N_ref:>7,}): {time.perf_counter()-t0:.1f}s')

t_table = time.perf_counter() - t0_table
print(f'\nTable 1 computed in {t_table:.1f}s  [{DEVICE}]  (baseline: ~36 min CPU)')

# Print the table
print(f"\n{'K':>3} |", end='')
for N in N_vals_bm: print(f' {N:>8}', end='')
print()
print('-' * (5 + 9*len(N_vals_bm)))
for ki, K in enumerate(K_vals_bm):
    print(f'{K:3d} |', end='')
    for ni in range(len(N_vals_bm)):
        v = table1[ki, ni]
        print(f' {v:>8.2f}' if v < 1000 else f' {v:>8.0f}', end='')
    print()
print('-' * (5 + 9*len(N_vals_bm)))
print('K*  |', end='')
for N in N_vals_bm: print(f' {np.log(N)/c_rho:>8.1f}', end='')
print()

In [ ]:
K_arr = np.array(K_vals_bm)
N_arr_bm = np.array(N_vals_bm, dtype=float)

fig = plt.figure(figsize=(16, 6))
gs  = gridspec.GridSpec(1, 2, wspace=0.35)

# Heatmap
ax1 = fig.add_subplot(gs[0, 0])
t1c = np.clip(table1, 1e-4, None)
im  = ax1.imshow(t1c, aspect='auto', cmap='RdYlGn_r',
                 norm=LogNorm(vmin=0.01, vmax=max(1e4, t1c.max())),
                 origin='lower', interpolation='nearest')
ax1.set_xticks(range(len(N_vals_bm)))
ax1.set_xticklabels([str(N) for N in N_vals_bm], rotation=45, fontsize=8)
ax1.set_yticks(range(len(K_vals_bm)))
ax1.set_yticklabels(K_vals_bm)
ax1.set_xlabel('$N$');  ax1.set_ylabel('$K$')
Nc  = np.linspace(0, len(N_vals_bm)-1, 200)
lNc = np.interp(Nc, range(len(N_vals_bm)), np.log(N_arr_bm))
ax1.plot(Nc, lNc/c_rho - K_vals_bm[0], 'k--', lw=2.5, label=r'$K^* = \log N/c_\rho$')
plt.colorbar(im, ax=ax1, label='MSE');  ax1.legend(fontsize=9)
ax1.set_title('Heatmap: Phase Transition (Normal Setting)')

# MSE vs N log-log
ax2 = fig.add_subplot(gs[0, 1])
cmap_tab = plt.get_cmap('tab10')
for ki, K in enumerate(K_vals_bm):
    style = '-o' if K <= 6 else '--s'
    ax2.loglog(N_arr_bm, table1[ki], style, ms=4,
               color=cmap_tab(ki/len(K_vals_bm)), label=f'K={K}')
ref_line = table1[2, -1] * N_arr_bm[-1] / N_arr_bm
ax2.loglog(N_arr_bm, ref_line, 'k--', lw=1.5, label=r'$O(1/N)$')
ax2.set_xlabel('$N$');  ax2.set_ylabel('MSE')
ax2.set_title('MSE vs $N$ for each $K$')
ax2.legend(fontsize=7, ncol=2)

fig.suptitle(f'G&Y Table 1 Reproduction — Normal Setting [{DEVICE}, B={B_BATCHES}]',
             fontsize=13, fontweight='bold')
plt.savefig('MB_reproduction.png', **SAVEFIG_KW);  plt.show()
print('Saved: MB_reproduction.png')

## 2.2  Verification of Lemmas 1–2

Direct Monte Carlo estimate of $\mathbb{E}[\psi_{2K}^2(S_2)\,\psi_{1K}^2(S_1)]$ vs the analytical formula (Lemma 1) and the exponential asymptotic $e^{c_\rho K}$ (Lemma 2 / Stirling).

In [ ]:
K_check_arr = np.arange(1, 9)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.semilogy(K_check_arr, moments_analytical, 'b-o', ms=6, label='Analytical (Lemma 1)')
ax.semilogy(K_check_arr, moments_mc, 'r-s', ms=5, label='Monte Carlo')
ax.semilogy(K_check_arr, np.exp(c_rho*K_check_arr), 'k--', lw=1.5, label=r'$e^{c_\rho K}$ bound')
ax.set_xlabel('K');  ax.set_ylabel(r'$E[\psi_{2K}^2 \psi_{1K}^2]$')
ax.set_title('Lemma 1: 4th-order Moments vs Analytical');  ax.legend(fontsize=9)

ax = axes[1]
ratio = np.array(moments_analytical) / np.exp(c_rho * K_check_arr)
ax.plot(K_check_arr, ratio, 'b-o', ms=6)
ax.set_xlabel('K');  ax.set_ylabel(r'Moment / $e^{c_\rho K}$')
ax.set_title('Polynomial Prefactor\n(Stirling correction from Lemma 2)')

fig.suptitle(r'Verification of Lemmas 1 & 2 ($t_1=1, t_2=2, \rho=2$)', fontweight='bold')
plt.tight_layout()
plt.savefig('lemma_verification.png', **SAVEFIG_KW);  plt.show()
print('Saved: lemma_verification.png')

## 2.3  MSE Table — Lognormal Setting ($t_1 = 1$, $t_2 = 2$)

Basis: $\psi_k(S(t)) = S(t)^k e^{-k^2 t/2}$ (martingale power basis). The Gram matrix has entries $\Psi_{jk} = e^{jkt}$ — Vandermonde-type with condition number $\sim e^{K^2}$, making the inversion numerically unreliable for $K \geq 5$.

Critical thresholds: $K_{\text{conv}} = \sqrt{\log N / 7}$, $K_{\text{div}} = \sqrt{\log N / 5}$.

In [ ]:
def compute_mse_lognormal_allN(K, t1, t2, N_ref, N_vals, B=3000, chunk_b=None):
    """
    Same one-pass-all-N trick for the lognormal / power-basis setting.
    psi_k(S(t)) = exp(k W(t) - k^2 t/2). Gram matrix Psi must be inverted.
    """
    if chunk_b is None:
        chunk_b = GPU_CHUNK if DEVICE == 'GPU' else 50

    idx     = np.arange(K+1)
    Psi     = np.exp(np.outer(idx, idx) * t1)
    try:     Psi_inv = np.linalg.inv(Psi)
    except:  return {N: np.inf for N in N_vals}

    Psi_inv_xp = xp.array(Psi_inv, dtype=xp.float64)
    bt_np  = np.zeros(K+1);  bt_np[K] = 1.0

    sum_g  = xp.zeros(K+1, dtype=xp.float64)
    sum_g2 = xp.zeros(K+1, dtype=xp.float64)

    if hasattr(xp, 'random') and hasattr(xp.random, 'default_rng'):
        rng_ln = xp.random.default_rng(123 + K)
        def gen(shape): return rng_ln.standard_normal(shape).astype(xp.float64)
    else:
        xp.random.seed(123 + K)
        def gen(shape): return xp.random.standard_normal(shape).astype(xp.float64)

    batches_done = 0
    while batches_done < B:
        b  = min(chunk_b, B - batches_done)
        W1 = gen((b, N_ref)) * np.sqrt(t1)          # (b, N_ref)
        W2 = W1 + gen((b, N_ref)) * np.sqrt(t2-t1)
        Y  = xp.exp(K*W2 - K*K*t2/2)               # (b, N_ref)

        gamma_hat = xp.zeros((b, K+1), dtype=xp.float64)
        for k in range(K+1):
            psi_k        = xp.exp(k*W1 - k*k*t1/2)
            gamma_hat[:, k] = (Y * psi_k).mean(axis=1)

        # beta_hat = Psi_inv @ gamma_hat  (batch via einsum)
        beta_hat = xp.einsum('ij,bj->bi', Psi_inv_xp, gamma_hat)  # (b, K+1)

        sum_g  += beta_hat.sum(axis=0)
        sum_g2 += (beta_hat**2).sum(axis=0)
        batches_done += b

    mean_b = sum_g / B
    var_b  = sum_g2 / B - mean_b**2
    mb_np  = to_numpy(mean_b)
    vb_np  = to_numpy(var_b)

    mse_dict = {}
    for N in N_vals:
        mse_dict[N] = float(((N_ref/N) * vb_np + (mb_np - bt_np)**2).sum())
    return mse_dict


N_vals_ln = [1000, 5000, 10000, 50000, 100000, 500000]
K_vals_ln = list(range(1, 7))
N_REF_LN  = 500_000

table2 = np.zeros((len(K_vals_ln), len(N_vals_ln)))
t0_ln  = time.perf_counter()
for ki, K in enumerate(K_vals_ln):
    t0 = time.perf_counter()
    mse_d = compute_mse_lognormal_allN(K, t1_gy, t2_gy, N_REF_LN, N_vals_ln, B=3000)
    for ni, N in enumerate(N_vals_ln):
        table2[ki, ni] = mse_d[N]
    print(f'  K={K}: {time.perf_counter()-t0:.1f}s')

print(f'\nLognormal table: {time.perf_counter()-t0_ln:.1f}s  [{DEVICE}]')

print(f"\n{'K':>3} |", end='')
for N in N_vals_ln: print(f' {N:>10}', end='')
print()
print('-' * (5 + 11*len(N_vals_ln)))
for ki, K in enumerate(K_vals_ln):
    print(f'{K:3d} |', end='')
    for ni in range(len(N_vals_ln)):
        v = table2[ki, ni]
        print(f' {v:>10.3f}' if v < 1000 else f' {v:>10.0f}', end='')
    print()
print('-' * (5 + 11*len(N_vals_ln)))
print('Kconv|', end='')
for N in N_vals_ln: print(f' {np.sqrt(np.log(N)/(5+2)):>10.2f}', end='')
print('  (sufficient)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

ax = axes[0]
t2c = np.clip(table2, 1e-4, None)
im2 = ax.imshow(t2c, aspect='auto', cmap='RdYlGn_r',
                norm=LogNorm(vmin=0.01, vmax=max(1e4, t2c.max())), origin='lower')
ax.set_xticks(range(len(N_vals_ln)))
ax.set_xticklabels([str(N) for N in N_vals_ln], rotation=45, fontsize=8)
ax.set_yticks(range(len(K_vals_ln)))
ax.set_yticklabels(K_vals_ln)
Nc_ln = np.linspace(0, len(N_vals_ln)-1, 200)
lNc_ln = np.interp(Nc_ln, range(len(N_vals_ln)), np.log(N_vals_ln))
ax.plot(Nc_ln, np.sqrt(lNc_ln/(5+2)) - K_vals_ln[0], 'b--', lw=2, label=r'$K_{\rm conv}$')
ax.plot(Nc_ln, np.sqrt(lNc_ln/(3+2)) - K_vals_ln[0], 'r--', lw=2, label=r'$K_{\rm div}$')
plt.colorbar(im2, ax=ax);  ax.legend(fontsize=9)
ax.set_xlabel('$N$');  ax.set_ylabel('$K$');  ax.set_title('Lognormal MSE Heatmap')

ax = axes[1]
N_ln_arr = np.array(N_vals_ln, dtype=float)
for ki, K in enumerate(K_vals_ln):
    ax.loglog(N_ln_arr, table2[ki], '-o', ms=4,
              color=plt.get_cmap('tab10')(ki/6), label=f'K={K}')
ax.loglog(N_ln_arr, table2[0,-1]*N_ln_arr[-1]/N_ln_arr, 'k--', lw=1.5, label=r'$O(1/N)$')
ax.set_xlabel('$N$');  ax.set_ylabel('MSE')
ax.set_title('Lognormal MSE vs $N$')
ax.legend(fontsize=8)

fig.suptitle(f'Theorem 2: Lognormal Setting [{DEVICE}, B=3000, N_ref={N_REF_LN:,}]',
             fontsize=13, fontweight='bold')
plt.savefig('lognormal_results.png', **SAVEFIG_KW);  plt.show()
print('Saved: lognormal_results.png')

## 2.4  Practical LSM: American Put RMSE Grid

Longstaff–Schwartz algorithm on an American put ($S_0 = K = 40$, $r = 0.06$, $\sigma = 0.2$, $T = 1$, $m = 50$ exercise dates) using weighted Laguerre polynomial basis functions, as in Longstaff & Schwartz (2001). Benchmark from a 5 000-step binomial tree. Each cell of the grid is the RMSE over 20 independent trials, parallelised with `joblib`.

In [ ]:
S0_put=40; K_put=40; r_put=0.06; sig_put=0.2; T_put=1.0; m_put=50
benchmark = binomial_american_put(S0_put, K_put, r_put, sig_put, T_put)
print(f'Benchmark: {benchmark:.4f}')

N_grid    = [500, 1000, 2000, 5000, 10000, 20000, 50000, 100000]
K_grid    = [2, 3, 4, 5, 6, 8, 10, 13]
N_TRIALS  = 20

def _lsm_trial(S0, Ks, r, sig, T, m, N, Kb, trial, ki, ni):
    p, _ = lsm_american_put(S0, Ks, r, sig, T, m, N, Kb,
                            seed=1000*trial + ni*100 + ki)
    return p

rmse_grid = np.zeros((len(K_grid), len(N_grid)))
bias_grid = np.zeros((len(K_grid), len(N_grid)))

t0_grid = time.perf_counter()
for ki, Kb in enumerate(K_grid):
    t0k = time.perf_counter()
    for ni, N in enumerate(N_grid):
        # Parallelise over N_TRIALS — each is embarrassingly independent
        prices = Parallel(n_jobs=-1, prefer='threads')(
            delayed(_lsm_trial)(S0_put, K_put, r_put, sig_put, T_put,
                                m_put, N, Kb, trial, ki, ni)
            for trial in range(N_TRIALS)
        )
        prices = np.array(prices)
        rmse_grid[ki, ni] = np.sqrt(np.mean((prices - benchmark)**2))
        bias_grid[ki, ni] = prices.mean() - benchmark
    print(f'  K={Kb:2d}: {time.perf_counter()-t0k:.1f}s')

print(f'\nLSM grid total: {time.perf_counter()-t0_grid:.1f}s  (parallel={N_TRIALS} trials/cell)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, grid, title, cmap_n in zip(
    axes, [rmse_grid, bias_grid],
    ['RMSE of LSM Price', 'Bias of LSM Price'],
    ['YlOrRd', 'RdBu']
):
    if cmap_n == 'YlOrRd':
        norm = LogNorm(vmin=0.01, vmax=grid.max())
    else:
        bmax = max(abs(grid.min()), abs(grid.max()))
        norm = matplotlib.colors.TwoSlopeNorm(vmin=-bmax, vcenter=0, vmax=bmax)
    im = ax.imshow(grid, aspect='auto', cmap=cmap_n, norm=norm,
                   origin='lower', interpolation='nearest')
    ax.set_xticks(range(len(N_grid)))
    ax.set_xticklabels([str(N) for N in N_grid], rotation=45, fontsize=8)
    ax.set_yticks(range(len(K_grid)))
    ax.set_yticklabels(K_grid)
    ax.set_xlabel('$N$');  ax.set_ylabel('$K$');  ax.set_title(title)
    plt.colorbar(im, ax=ax)
    for ki in range(len(K_grid)):
        for ni in range(len(N_grid)):
            ax.text(ni, ki, f'{grid[ki,ni]:.2f}', ha='center', va='center', fontsize=5.5)

fig.suptitle(f'Practical LSM — American Put (benchmark={benchmark:.4f}, {N_TRIALS} trials/cell)',
             fontsize=13, fontweight='bold')
plt.savefig('american_put_rmse.png', **SAVEFIG_KW);  plt.show()
print('Saved: american_put_rmse.png')

## 2.5  Theory vs Practice: Critical $K^*$ Bounds and Overfitting

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# K* bounds comparison
ax = axes[0]
N_r = np.logspace(3, 8, 300)
ax.plot(N_r, np.log(N_r)/c_rho, 'b-', lw=2.5,
        label=r'BM (Hermite): $K^* = \log N/c_\rho$  [Thm 1]')
ax.fill_between(N_r, np.sqrt(np.log(N_r)/(3+2)), np.sqrt(np.log(N_r)/(5+2)),
                alpha=0.3, color='red', label='GBM (power basis): critical band [Thm 2]')
ax.plot(N_r, N_r**(1/3), 'g--', lw=2, label=r'Stentoft (2004): $K = N^{1/3}$')
ax.axhspan(3, 5, alpha=0.12, color='orange', label='Practical sweet spot ($K=3$–$5$)')
ax.set_xscale('log');  ax.set_ylim(0, 12)
ax.set_xlabel('$N$');  ax.set_ylabel('$K^*$')
ax.set_title('Critical $K^*$ vs $N$: G&Y vs Stentoft');  ax.legend(fontsize=8)

# Overfitting K=3 vs K=15
ax = axes[1]
N_ov = [500, 1000, 2000, 5000, 10000, 50000, 100000]
n_ov = 30

for Kb, col, mk in [(3, 'steelblue', 'o'), (15, 'tomato', 's')]:
    rmse_ov = []
    for N_o in N_ov:
        # 1. On lance Parallel sans le [0] à la fin
        results = Parallel(n_jobs=-1, prefer='threads')(
            delayed(lsm_american_put)(S0_put, K_put, r_put, sig_put, T_put,
                                     m_put, N_o, Kb, seed=t*137+1)
            for t in range(n_ov)
        )
        # 2. results est une liste de tuples (prix, se). On extrait les prix :
        prices = [res[0] for res in results]

        # 3. On calcule la RMSE sur ces prix extraits
        rmse_ov.append(np.sqrt(np.mean((np.array(prices) - benchmark)**2)))

    ax.loglog(N_ov, rmse_ov, f'{mk}-', color=col, ms=5, label=f'$K={Kb}$')

ax.set_xlabel('$N$');  ax.set_ylabel('RMSE')
ax.set_title('Overfitting: $K=3$ (robust) vs $K=15$ (overfit)');  ax.legend()

fig.suptitle('Part 2: Theory vs Practice', fontweight='bold')
plt.tight_layout()
plt.savefig('theory_vs_practice.png', dpi=150, bbox_inches='tight');  plt.show()
print('Saved: theory_vs_practice.png')